# 98 Production Readiness Gate

Run this notebook to get a strict decision:
- `PILOT_READY: YES/NO`
- `FULL_PRODUCTION_READY: YES/NO`


In [ ]:
from pathlib import Path
import pandas as pd

REPORTS = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')

PATHS = {
    'model_summary': REPORTS / 'portable_m5_transfer_hcap_tuned_summary.csv',
    'model_metrics': REPORTS / 'portable_m5_transfer_hcap_tuned_metrics.csv',
    'naive_summary': REPORTS / 'm5_dept_store_summary.csv',
}

THRESHOLDS = {
    'min_relative_wape_improvement': 0.05,
    'min_relative_rmse_improvement': 0.05,
    'max_h12_wape': 0.165,
    'max_under_forecast_rate': 0.75,
}

# Manual gates: update these after technical/business validation.
MANUAL_GATES = {
    'api_contract_passed': False,
    'artifact_integrity_passed': False,
    'runtime_reliability_passed': False,
    'observability_passed': False,
    'sla_load_test_passed': False,
    'security_governance_passed': False,
    'deployment_safety_passed': False,
    'business_kpi_signoff_passed': False,
    'decision_policy_passed': False,
}

for _, v in PATHS.items():
    if not v.exists():
        raise FileNotFoundError('Missing required file: {}'.format(v))

print('Input files found')


In [ ]:
model_summary = pd.read_csv(PATHS['model_summary']).iloc[0]
naive_summary = pd.read_csv(PATHS['naive_summary']).iloc[0]
model_metrics = pd.read_csv(PATHS['model_metrics'])

m_test = model_metrics[(model_metrics['split'] == 'test') & (model_metrics['horizon'] > 0)].copy()
h912 = m_test[m_test['horizon'].isin([9, 10, 11, 12])]

rel_wape_impr = (naive_summary['WAPE'] - model_summary['WAPE']) / naive_summary['WAPE']
rel_rmse_impr = (naive_summary['RMSE'] - model_summary['RMSE']) / naive_summary['RMSE']

checks = []
checks.append({
    'gate': 'A1_wape_vs_naive',
    'passed': bool(model_summary['WAPE'] < naive_summary['WAPE']),
    'value': float(model_summary['WAPE']),
    'target': '< {:.6f}'.format(float(naive_summary['WAPE'])),
})
checks.append({
    'gate': 'A1_rmse_vs_naive',
    'passed': bool(model_summary['RMSE'] < naive_summary['RMSE']),
    'value': float(model_summary['RMSE']),
    'target': '< {:.3f}'.format(float(naive_summary['RMSE'])),
})
checks.append({
    'gate': 'A1_mase_vs_naive',
    'passed': bool(model_summary['MASE_mean'] < naive_summary['MASE_mean']),
    'value': float(model_summary['MASE_mean']),
    'target': '< {:.6f}'.format(float(naive_summary['MASE_mean'])),
})
checks.append({
    'gate': 'A1_rel_wape_improvement',
    'passed': bool(rel_wape_impr >= THRESHOLDS['min_relative_wape_improvement']),
    'value': float(rel_wape_impr),
    'target': '>= {:.3f}'.format(THRESHOLDS['min_relative_wape_improvement']),
})
checks.append({
    'gate': 'A1_rel_rmse_improvement',
    'passed': bool(rel_rmse_impr >= THRESHOLDS['min_relative_rmse_improvement']),
    'value': float(rel_rmse_impr),
    'target': '>= {:.3f}'.format(THRESHOLDS['min_relative_rmse_improvement']),
})

h12 = h912[h912['horizon'] == 12]
h12_wape = float(h12['WAPE'].iloc[0]) if not h12.empty else float('nan')
checks.append({
    'gate': 'A2_h12_wape_limit',
    'passed': bool(h12_wape <= THRESHOLDS['max_h12_wape']),
    'value': h12_wape,
    'target': '<= {:.3f}'.format(THRESHOLDS['max_h12_wape']),
})

under = float(model_summary['under_forecast_rate'])
checks.append({
    'gate': 'A4_under_forecast_rate_limit',
    'passed': bool(under <= THRESHOLDS['max_under_forecast_rate']),
    'value': under,
    'target': '<= {:.3f}'.format(THRESHOLDS['max_under_forecast_rate']),
})

gate_df = pd.DataFrame(checks)
gate_df


In [ ]:
ds_must = {
    'A1_wape_vs_naive',
    'A1_rmse_vs_naive',
    'A1_mase_vs_naive',
    'A1_rel_wape_improvement',
    'A1_rel_rmse_improvement',
    'A2_h12_wape_limit',
    'A4_under_forecast_rate_limit',
}
ds_pass = bool(gate_df[gate_df['gate'].isin(ds_must)]['passed'].all())
manual_df = pd.DataFrame([{'gate': k, 'passed': bool(v)} for k, v in MANUAL_GATES.items()])
manual_pass = bool(manual_df['passed'].all())

pilot_ready = ds_pass
full_ready = ds_pass and manual_pass

print('PILOT_READY:', 'YES' if pilot_ready else 'NO')
print('FULL_PRODUCTION_READY:', 'YES' if full_ready else 'NO')
print('DS_GATES_PASS:', ds_pass)
print('MANUAL_TECH_BUSINESS_GATES_PASS:', manual_pass)

display(gate_df)
display(manual_df)


## How to use

1. Run notebook `05b` first.
2. Run this notebook.
3. If DS gates pass, pilot is ready.
4. Set `MANUAL_GATES` true only after technical/business sign-off.
5. Full production is YES only when DS + manual gates both pass.
